In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import Image


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [3]:
transform = transforms.Compose([
    transforms.Resize((96, 96)),   # smaller ghetoy fast pahije mhnun 
    transforms.ToTensor(),
    #transforms.Normalize([0.5], [0.5])
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])

])


In [4]:
train_dir = "dataset/train"

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print("Classes:", train_dataset.classes)


Classes: ['AbnormalHB', 'Normal', 'historyofmi', 'mi']


In [5]:
class ECG_CNN(nn.Module):
    def __init__(self):
        super(ECG_CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.AdaptiveAvgPool2d((4,4))
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 4)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x


In [6]:
model = ECG_CNN().to(device)


In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)



In [9]:
epochs = 11

for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss:.4f} Accuracy: {accuracy:.2f}%")


Epoch [1/11] Loss: 38.4582 Accuracy: 36.21%
Epoch [2/11] Loss: 34.0100 Accuracy: 51.40%
Epoch [3/11] Loss: 31.1653 Accuracy: 57.33%
Epoch [4/11] Loss: 28.2279 Accuracy: 62.28%
Epoch [5/11] Loss: 25.8258 Accuracy: 68.21%
Epoch [6/11] Loss: 23.4614 Accuracy: 71.98%
Epoch [7/11] Loss: 20.3348 Accuracy: 76.62%
Epoch [8/11] Loss: 18.2001 Accuracy: 78.23%
Epoch [9/11] Loss: 15.2977 Accuracy: 82.65%
Epoch [10/11] Loss: 13.4518 Accuracy: 86.96%
Epoch [11/11] Loss: 11.4663 Accuracy: 89.33%


In [10]:
torch.save(model.state_dict(), "ecg_cnn_model.pth")
print("Model Saved Successfully!")




Model Saved Successfully!


In [11]:
train_loader_eval = DataLoader(train_dataset, batch_size=32, shuffle=False)


In [12]:
# -----------------------------
# Calculate Overall Training Accuracy
# -----------------------------
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

overall_train_accuracy = 100 * correct / total

print("\n==============================")
print(f"Overall Training Accuracy: {overall_train_accuracy:.2f}%")
print("==============================")



Overall Training Accuracy: 92.89%


In [13]:
# model la load process
model = ECG_CNN().to(device)
model.load_state_dict(torch.load("ecg_cnn_model.pth", map_location=device))
model.eval()

# Give image path
# image_path = "dataset/train/Normal/Normal(17).jpg"
image_path = r"C:\Users\Shubham\100-days-ml\ML\ECG\dataset\train\mi\MI_Patient(267).jpg"
#image_path=r"C:\Users\Shubham\100-days-ml\ML\ECG\dataset\train\historyofmi\MI_History_Patient(129).jpg"

image = Image.open(image_path).convert("RGB")
image = transform(image)
image = image.unsqueeze(0).to(device)

with torch.no_grad():
    outputs = model(image)
    _, predicted = torch.max(outputs, 1)

classes = train_dataset.classes
print("Prediction:", classes[predicted.item()])


Prediction: mi
